# Train the SAC irrigation controller

Parameter-shared actor + twin VDN LayerNorm critic; weak entropy pin and a two-phase exploration-noise schedule. Best model selected on the held-out dev set.

In [ ]:
# Clone the repo and install the RL stack (Colab / Kaggle GPU).
import os, subprocess, sys
REPO = '/content/thesis'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/taratorbati/thesis.git', REPO], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
subprocess.run(['pip', 'install', '--quiet',
                'stable-baselines3>=2.6.0', 'gymnasium', 'wandb', 'pytest'], check=True)
import torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# Optional: Weights & Biases logging (Enter to skip).
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY'); print('W&B key loaded.')
except Exception:
    import getpass
    k = getpass.getpass('WANDB key (Enter to skip): ').strip()
    if k:
        os.environ['WANDB_API_KEY'] = k
    else:
        print('Skipping W&B.')

In [ ]:
# Pre-flight smoke tests.
import subprocess, sys
assert subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-q']).returncode == 0, 'TESTS FAILED'

In [ ]:
# Train SAC (the v2.18-p3b configuration), 250k steps (~30-55 min A100 / ~2-2.5 h T4).
SEED = 0
from src.rl.train_sac import train_sac
model = train_sac(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',   # or None
    total_timesteps=250_000,
)

In [ ]:
# Evaluate the best checkpoint over the 9-cell grid (perfect forecast).
import glob, os, subprocess, sys
run = sorted(glob.glob(f'/content/thesis/results/rl/sac_seed{SEED}_*'), key=os.path.getmtime)[-1]
best = os.path.join(run, 'best_model', 'best_model.zip')
subprocess.run([sys.executable, '-m', 'scripts.experiments.exp_rl',
                '--mode', 'eval', '--model', best], check=True)
print('Evaluated', best)

In [ ]:
# Optional (Colab): archive the run to Google Drive.
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, os
# dst = '/content/drive/MyDrive/thesis_runs/' + os.path.basename(run)
# shutil.copytree(run, dst, ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
# print('Archived to', dst)